### 1.) Parâmetros do ambiente

In [0]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "cine_data_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Inputs"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

catalog: cine_data_medallion
bronze_schema: cine_data_medallion.bronze
silver_schema: cine_data_medallion.silver
gold_schema: cine_data_medallion.gold
landing_path: /Volumes/cine_data_medallion/bronze/Inputs


### 2.) Leitura dos arquivos csv armazenados no volume

In [0]:
from pyspark.sql.functions import current_timestamp

# 1. Mapeamento dos caminhos no Volume (criado no notebook 00, schema landing)
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

# 2. Leitura pura (sem transformações de negócio)
df_credits_tags_raw = spark.read.csv(path_credits_and_tags, header=True, inferSchema=True)
df_financials_raw = spark.read.csv(path_movies_financials, header=True, inferSchema=True)
df_info_raw = spark.read.csv(path_movies_info, header=True, inferSchema=True)
df_metrics_raw = spark.read.csv(path_movies_metrics, header=True, inferSchema=True)
df_reviews_raw = spark.read.csv(path_movies_reviews, header=True, inferSchema=True)

In [0]:
# 3. Gravação com adição do timestamp no momento da escrita
df_credits_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_credits_and_tags")

df_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_financials")

df_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_info")

df_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_metrics")

df_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_reviews")


display(spark.table(f"{catalog}.bronze.tb_movies_info").limit(5))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-07T18:20:58.781Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-07T18:20:58.781Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-07T18:20:58.781Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-07T18:20:58.781Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-07T18:20:58.781Z


In [0]:
import requests
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Widgets: padrão = últimos 7 dias corridos a partir da execução
hoje = datetime.now()
dbutils.widgets.text("data_inicio", (hoje - timedelta(days=7)).strftime("%m-%d-%Y"), "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", hoje.strftime("%m-%d-%Y"), "Data fim (MM-DD-AAAA)")

data_inicio = datetime.strptime(dbutils.widgets.get("data_inicio"), "%m-%d-%Y")
data_fim = datetime.strptime(dbutils.widgets.get("data_fim"), "%m-%d-%Y")
print(data_inicio, data_fim)


# 2. Chamada à API
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo("
    "dataInicial=@dataInicial,"
    "dataFinalCotacao=@dataFinalCotacao)"
)

params ={
    "@dataInicial": f"'{data_inicio:%m-%d-%Y}'",
    "@dataFinalCotacao": f"'{data_fim:%m-%d-%Y}'",
    "$format": "json",
    "orderby": "dataHoraCotacao asc",
    "$top": 1000,
    "$skip": 0
}
response = requests.get(url, params=params, timeout=60)
registros = response.json()["value"]

print(f"{len(registros)} cotações retornadas entre {data_inicio:%d/%m/%Y} e {data_fim:%d/%m/%Y}")

# 3. DataFrame Spark com schema explícito
schema = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True),
])

df_cotacao = (
    spark.createDataFrame(registros, schema=schema)
    .withColumn("data_cotacao", F.to_date("dataHoraCotacao"))
    .withColumn("dt_ingestao", F.current_timestamp())
)

display(df_cotacao)

# 4. Gravação na camada bronze
path_table = f"{bronze_schema}.tb_cotacao_dolar"

if df_cotacao.isEmpty():
    print("Nenhuma cotação no período. Nada a gravar.")
else:
    df_cotacao.write.format("delta").mode("append").saveAsTable(path_table)


spark.table(path_table).orderBy(F.desc("dataHoraCotacao")).show(10, truncate=False)

2026-09-30 00:00:00 2026-10-07 00:00:00
6 cotações retornadas entre 30/09/2026 e 07/10/2026


dataHoraCotacao,cotacaoCompra,data_cotacao,dt_ingestao
2026-09-30 13:11:44.783262,5.1803,2026-09-30,2026-10-07T18:21:33.956Z
2026-10-01 13:10:35.4469,5.2073,2026-10-01,2026-10-07T18:21:33.956Z
2026-10-02 13:03:16.256632,5.2232,2026-10-02,2026-10-07T18:21:33.956Z
2026-10-05 13:07:36.558602,4.9853,2026-10-05,2026-10-07T18:21:33.956Z
2026-10-06 13:03:21.267287,4.9692,2026-10-06,2026-10-07T18:21:33.956Z
2026-10-07 13:05:23.249063,4.9929,2026-10-07,2026-10-07T18:21:33.956Z


+--------------------------+-------------+------------+--------------------------+
|dataHoraCotacao           |cotacaoCompra|data_cotacao|dt_ingestao               |
+--------------------------+-------------+------------+--------------------------+
|2026-10-07 13:05:23.249063|4.9929       |2026-10-07  |2026-10-07 18:21:35.479136|
|2026-10-06 13:03:21.267287|4.9692       |2026-10-06  |2026-10-07 18:21:35.479136|
|2026-10-05 13:07:36.558602|4.9853       |2026-10-05  |2026-10-07 18:21:35.479136|
|2026-10-02 13:03:16.256632|5.2232       |2026-10-02  |2026-10-07 18:21:35.479136|
|2026-10-01 13:10:35.4469  |5.2073       |2026-10-01  |2026-10-07 18:21:35.479136|
|2026-09-30 13:11:44.783262|5.1803       |2026-09-30  |2026-10-07 18:21:35.479136|
+--------------------------+-------------+------------+--------------------------+

